# colab_12 — Geneformer CPT evals #1 & #2 (aggregated regime, v2 · seed 0)

Does the aggregated CPT run (colab_11 v2) actually **help the biology**, not just move the embedding? This notebook runs the two pre-registered evals that read the glia embedding directly:

- **Eval #1** — substate linear probe (does CPT sharpen within-lineage state structure?)
- **Eval #2** — APOE-carrier recovery within astrocytes and within microglia (the Stanton-core axis)

Both are scored strictly by `docs/EVALUATION_CONTRACT.md`: improvement over the zero-shot baseline, on the **same global held-out donors**, against bands fixed before any CPT result existed.

**Two extraction points, by design.** The head-ablation diagnostic showed v2's loss gain is concentrated in the head + last encoder layer — structurally *downstream* of the standard cell-embedding readout (`emb_layer=-1`, second-to-last layer). So every eval here is run at **both** `emb_layer=-1` (the readout the whole pipeline uses) **and** `emb_layer=0` (the last layer). The −1-vs-0 gap is the eval-space read on layer 11's absorbed share — a partial view: it captures layer 11 but *not* the head, whose contribution lives only in reconstruction space and is unreachable as an embedding.

Eval #3 / detector #2 (catastrophic forgetting) are **not** here — they need a non-AD reference dataset and are a separate deliverable (colab_13).

## 1 — Setup

### 1a — Mount Drive, clone/pull the repo, install the Geneformer environment, set run flags

Same lean Geneformer-only stack as colab_09/11 (rides Colab's native numpy-2 base; scGPT is not installed). A GPU is required — this notebook re-embeds the substrate, which is not viable on CPU.

`SMOKE` is the one live switch: committed `False`. When `True`, the notebook subsamples the substrate to a few thousand cells *after* the split is assigned, so the whole path (tokenize → embed at layer 0 → probe → k-NN) can be exercised cheaply before paying for the full embedding. Every derived path is suffixed `_SMOKE` so a smoke run can waste time but never overwrite a real output.

In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = "/content/drive/MyDrive/ad-glia-fm-prep"
os.makedirs(DRIVE_ROOT, exist_ok=True)

REPO_URL  = "https://github.com/pavlemic/ad-glia-fm-prep.git"
REPO_PATH = "/content/ad-glia-fm-prep"
if not os.path.exists(REPO_PATH):
    subprocess.run(["git", "clone", REPO_URL, REPO_PATH], check=True)
else:
    subprocess.run(["git", "-C", REPO_PATH, "pull"], check=True)
if REPO_PATH not in sys.path:
    sys.path.insert(0, REPO_PATH)

assert sys.version_info >= (3, 10), f"Geneformer needs Python >=3.10, got {sys.version_info[:2]}."

!pip install -r {REPO_PATH}/requirements_geneformer.txt

GENEFORMER_REPO = "/content/Geneformer"
if not os.path.exists(GENEFORMER_REPO):
    !git lfs install
    !git clone https://huggingface.co/ctheodoris/Geneformer {GENEFORMER_REPO}
!cd {GENEFORMER_REPO} && pip install .
# torchao (Colab-preinstalled, unused) hard-raises inside peft dispatch below its floor -- remove it.
!pip uninstall -y torchao -q

GENEFORMER_COMMIT = subprocess.run(
    ["git", "-C", GENEFORMER_REPO, "rev-parse", "HEAD"],
    capture_output=True, text=True).stdout.strip()

import torch
assert torch.cuda.is_available(), "no CUDA GPU -- select a GPU runtime before re-embedding"
print("Python:", sys.version.split()[0], "| Geneformer commit:", GENEFORMER_COMMIT,
      "| GPU:", torch.cuda.get_device_name(0))

# --- the one live switch; ALL variable output paths derive from SUFFIX ---
SMOKE   = False
SMOKE_N_PER_GROUP = 40          # cells kept per (lineage x split x substate) group when SMOKE
SUFFIX  = "_SMOKE" if SMOKE else ""
SEED    = 0
from datetime import date
TODAY   = date.today().isoformat()
print(f"SMOKE={SMOKE} | output suffix='{SUFFIX}'")

## 2 — Load the substrate and validate the schema

### 2a — Rebuild the glia substrate (deterministic; same `cell_index` as the saved embeddings)

The saved embeddings carry a `cell_index` that was assigned when colab_11 concatenated the microglia and astrocyte subsets. Rebuilding `glia` here from the *same* two subset files, in the *same* order, reproduces that `cell_index` exactly — it is the key every embedding matrix is realigned to. We need the raw-counts object (not just the embeddings) because re-tokenizing for the layer-0 pass requires the counts.

`region` is kept here directly: colab_07/08 built each subset as a `.copy()` of the colab_05 parent without dropping obs columns, so `region` (needed for eval #2's confound audit) already rides along on the subset files — no separate join required.

In [ ]:
import gc
import numpy as np
import pandas as pd
import anndata as ad
import scanpy as sc
import scipy.sparse as sp
sc.settings.verbosity = 1

MICRO_PATH = os.path.join(DRIVE_ROOT, "micro_subset", "micro_subset.h5ad")
ASTRO_PATH = os.path.join(DRIVE_ROOT, "astro_subset", "astro_subset.h5ad")
for p in (MICRO_PATH, ASTRO_PATH):
    if not os.path.exists(p):
        raise FileNotFoundError(f"missing labelled subset {p} (colab_07 / colab_08 output)")

micro = sc.read_h5ad(MICRO_PATH)
astro = sc.read_h5ad(ASTRO_PATH)
assert list(micro.var_names) == list(astro.var_names), "gene panels differ between subsets"
micro.obs["lineage"] = "microglia"
astro.obs["lineage"] = "astrocyte"
KEEP_OBS = ["lineage", "substate", "apoe_carrier", "study_id", "donor_id", "region", "total_counts"]
micro.obs = micro.obs[[c for c in KEEP_OBS if c in micro.obs.columns]].copy()
astro.obs = astro.obs[[c for c in KEEP_OBS if c in astro.obs.columns]].copy()
glia = ad.concat([micro, astro], join="inner", index_unique="-")
del micro, astro; gc.collect()
glia.obs["cell_index"] = np.arange(glia.n_obs)
print("combined glia:", glia.shape)
print("lineage:", glia.obs["lineage"].value_counts().to_dict())
print("substate:", glia.obs["substate"].value_counts(dropna=False).to_dict())
print("apoe_carrier:", glia.obs["apoe_carrier"].value_counts(dropna=False).to_dict())

# raw-counts guard -- Geneformer rank-encodes RAW counts.
_idx = np.random.default_rng(0).choice(glia.n_obs, size=min(2000, glia.n_obs), replace=False)
Xs = glia.X[_idx]
data = Xs.data if sp.issparse(Xs) else np.asarray(Xs).ravel()
frac_int = float(np.mean(np.mod(data, 1) == 0)) if data.size else 1.0
assert frac_int >= 0.99, f".X is not raw counts (int frac {frac_int:.3f})"
assert glia.n_obs == 142588, f"expected 142,588-cell substrate, got {glia.n_obs} -- subset files changed"

### 2b — Fail loud on the obs schema eval #1/#2 depend on

Eval #2's contract *mandates* reporting the study **and region** composition of the cells driving any APOE recovery, and eval #1 needs `substate`. This cell asserts every such column is present and non-null and that the categorical values are the expected ones — so a missing or malformed label stops the run here instead of silently degrading a downstream audit. `region` in particular must have survived the subset inheritance (§2a); if it is absent or partly null, the assert points back to the colab_07/08 outputs.

In [ ]:
REQUIRED_OBS = ["cell_index", "lineage", "substate", "apoe_carrier", "study_id", "region", "donor_id"]
missing_cols = [c for c in REQUIRED_OBS if c not in glia.obs.columns]
assert not missing_cols, (
    f"substrate missing required obs columns: {missing_cols} "
    "(region is inherited from the colab_07/08 subsets -- check they carry it)")

for col in REQUIRED_OBS:
    n_null = int(pd.isna(glia.obs[col]).sum())
    assert n_null == 0, f"{col} has {n_null} null values -- eval audits require it complete"

assert set(glia.obs["lineage"]) == {"microglia", "astrocyte"}, "unexpected lineage values"
assert set(glia.obs["substate"]) <= {"homeostatic", "activated", "resting", "reactive", "intermediate"}, \
    f"unexpected substate values: {set(glia.obs['substate'])}"
assert set(glia.obs["apoe_carrier"]) <= {"carrier", "noncarrier", "e2"}, \
    f"unexpected apoe_carrier values: {set(glia.obs['apoe_carrier'])}"
print("obs schema OK | region x study:")
print(pd.crosstab(glia.obs["region"], glia.obs["study_id"]))

## 3 — Held-out split verification

### 3a — Rebuild the frozen donor split and hard-stop on any mismatch

The evals must score on the *same* held-out donors every regime and FM uses. The split is a deterministic function of the unchanged colab_05 parent (study-stratified `train_test_split`, seed searched over `range(200)` maximizing the worst-case test-donor margin), so rebuilding it here must reproduce the reference exactly. Any drift in seed, margin, or split counts is a hard stop — never a silently re-drawn split. Reference (colab_11, 2026-07-10): seed 32, worst-case margin 10, donors 101/22/22, cells 94963/23824/23801, test study fractions SEA-AD 0.567 / Li2025 0.30 / Haney2024 0.133.

If `SMOKE`, the subsample is drawn **here, after** the split is assigned, stratified by (lineage × split × substate) so every group the evals need stays populated.

In [ ]:
import json
from sklearn.model_selection import train_test_split

DONOR_META_PATH = os.path.join(REPO_PATH, "outputs", "donor_metadata.csv")
SPLIT_FRACS     = {"train": 0.70, "val": 0.15, "test": 0.15}
SEED_POOL       = range(200)

assert os.path.exists(DONOR_META_PATH), f"need committed donor metadata {DONOR_META_PATH} (colab_11 minted it)"
donor_meta = pd.read_csv(DONOR_META_PATH, dtype=str)
sub_donors = set(glia.obs["donor_id"].astype(str))
donor_meta = donor_meta[donor_meta["donor_id"].isin(sub_donors)].reset_index(drop=True)
assert donor_meta["donor_id"].is_unique, "a donor_id maps to >1 metadata row"

def _study_split(seed):
    d_tr, d_te = train_test_split(donor_meta["donor_id"], test_size=SPLIT_FRACS["test"],
                                  random_state=seed, stratify=donor_meta["study_id"])
    strat_tr = donor_meta.set_index("donor_id").loc[d_tr.values, "study_id"]
    d_tr, d_va = train_test_split(d_tr, test_size=SPLIT_FRACS["val"] / (SPLIT_FRACS["train"] + SPLIT_FRACS["val"]),
                                  random_state=seed, stratify=strat_tr)
    return set(d_tr), set(d_va), set(d_te)

def _test_margin(d_test):
    m = donor_meta[donor_meta["donor_id"].isin(d_test)]
    return int(min((m["apoe_carrier"] == "carrier").sum(), (m["apoe_carrier"] == "noncarrier").sum(),
                   (m["diagnosis"] == "AD").sum(), (m["diagnosis"] == "Control").sum()))

best_seed = max(SEED_POOL, key=lambda s: _test_margin(_study_split(s)[2]))
d_train, d_val, d_test = _study_split(best_seed)
margin = _test_margin(d_test)
split_map = {**{d: "train" for d in d_train}, **{d: "val" for d in d_val}, **{d: "test" for d in d_test}}
glia.obs["split"] = glia.obs["donor_id"].astype(str).map(split_map).astype("category")
assert not glia.obs["split"].isna().any(), "some cells' donor received no split assignment"

n_donors = {k: int(sum(1 for v in split_map.values() if v == k)) for k in ("train", "val", "test")}
n_cells  = glia.obs["split"].value_counts().to_dict()
test_by_study = glia.obs.loc[glia.obs["split"] == "test", "study_id"].value_counts(normalize=True).round(3).to_dict()
print(f"seed {best_seed} | margin {margin} | donors {n_donors} | cells {n_cells}")
print("test study fractions:", test_by_study)

# HARD STOP: match the frozen reference exactly (standing split-verification rule)
assert best_seed == 32,  f"seed {best_seed} != reference 32 -- split drifted, do NOT proceed"
assert margin == 10,     f"margin {margin} != reference 10 -- split drifted"
assert n_donors == {"train": 101, "val": 22, "test": 22}, f"donor counts {n_donors} != reference"
assert n_cells.get("train") == 94963 and n_cells.get("val") == 23824 and n_cells.get("test") == 23801, \
    f"cell counts {n_cells} != reference 94963/23824/23801"
print("split verification PASSED -- matches the frozen reference")

# --- SMOKE subsample: AFTER the split is assigned; keep every (lineage x split x substate) group ---
if SMOKE:
    keep = (glia.obs.groupby(["lineage", "split", "substate"], observed=True)
            .apply(lambda g: g.sample(min(len(g), SMOKE_N_PER_GROUP), random_state=SEED))
            .index.get_level_values(-1))
    glia = glia[glia.obs.index.isin(keep)].copy()
    print(f"[SMOKE] subsampled to {glia.n_obs} cells | split:", glia.obs["split"].value_counts().to_dict())

## 4 — Produce the last-layer (`emb_layer=0`) embeddings

### 4a — Tokenize the substrate (deterministic; identical encoding to the CPT run)

The layer-0 embeddings do not exist yet — colab_11 saved only the layer −1 embeddings and the LoRA adapter, and the tokenized dataset was ephemeral. So the substrate is re-tokenized here with the exact same encoding colab_09/11 used (V2, 4096 input size, GC104M dictionaries), gated on APOE surviving the vocabulary. Re-tokenizing is deterministic, so it reproduces the CPT run's inputs cell-for-cell.

In [ ]:
from geneformer import ENSEMBL_DICTIONARY_FILE, TOKEN_DICTIONARY_FILE, TranscriptomeTokenizer
import pickle
import geneformer.tokenizer as _gftok

with open(ENSEMBL_DICTIONARY_FILE, "rb") as f:
    symbol_to_ensembl = pickle.load(f)
with open(TOKEN_DICTIONARY_FILE, "rb") as f:
    token_dictionary = pickle.load(f)

glia.var["ensembl_id"] = [symbol_to_ensembl.get(s) for s in glia.var_names]
in_vocab = glia.var["ensembl_id"].map(lambda e: (e in token_dictionary) if e is not None else False)
apoe_e = symbol_to_ensembl.get("APOE")
assert apoe_e is not None and apoe_e in token_dictionary, (
    "APOE not tokenizable -- eval #2 impossible for Geneformer (pre-registered hard fail)")
print(f"genes in Geneformer vocab: {int(in_vocab.sum())} / {glia.n_vars}")

glia.obs["n_counts"] = np.asarray(glia.X.sum(axis=1)).ravel()
assert (glia.obs["n_counts"] > 0).all(), "cells with zero counts present"

TOK_IN_DIR  = f"/content/gf_token_in{SUFFIX}"
TOK_OUT_DIR = f"/content/gf_token_out{SUFFIX}"
os.makedirs(TOK_IN_DIR, exist_ok=True); os.makedirs(TOK_OUT_DIR, exist_ok=True)
glia.write_h5ad(os.path.join(TOK_IN_DIR, "glia.h5ad"))

CUSTOM_ATTRS = {c: c for c in ["cell_index", "split", "lineage", "substate", "apoe_carrier", "study_id", "donor_id"]}

# same RangeIndex monkeypatch colab_11 needed: tokenize_anndata positional-indexes var by an
# integer array against a non-integer index; reset it so upstream's assumed behaviour holds.
_orig_sum = _gftok.sum_ensembl_ids
def _sum_rangeindex_patch(*a, **k):
    r = _orig_sum(*a, **k); r.var.index = pd.RangeIndex(r.n_vars); return r
_gftok.sum_ensembl_ids = _sum_rangeindex_patch

tk = TranscriptomeTokenizer(CUSTOM_ATTRS, nproc=4)
tk.tokenize_data(TOK_IN_DIR, TOK_OUT_DIR, f"glia_evals{SUFFIX}", file_format="h5ad")
TOKENIZED_DATASET = os.path.join(TOK_OUT_DIR, f"glia_evals{SUFFIX}.dataset")
print("tokenized ->", TOKENIZED_DATASET)

### 4b — Extract layer-0 embeddings: frozen base (zero-shot L0) and merged v2 adapter (CPT L0)

Two passes of Geneformer's `EmbExtractor` at `emb_layer=0` (the last encoder layer). The zero-shot L0 runs the frozen base model; the CPT L0 reloads that same base, attaches the saved v2 LoRA adapter, and merges it before extracting — reproducing colab_11's post-CPT model, read one layer deeper than detector #1 did. Each output is written `_L0`-suffixed and guarded: if the file already exists (a re-run), it is loaded rather than recomputed.

In [ ]:
from peft import PeftModel
from transformers import BertForMaskedLM
from geneformer import EmbExtractor

GF_DIR    = os.path.join(DRIVE_ROOT, "geneformer")
MODEL_DIR = os.path.join(GENEFORMER_REPO, "Geneformer-V2-104M")
ADAPTER_DIR = os.path.join(GF_DIR, "cpt_aggregated_v2_seed0_adapter")   # colab_11 v2 output
assert os.path.exists(MODEL_DIR),   f"base model missing: {MODEL_DIR}"
assert os.path.exists(ADAPTER_DIR), f"v2 LoRA adapter missing on Drive: {ADAPTER_DIR}"

EMB_LABELS = ["cell_index", "split", "lineage", "substate", "apoe_carrier", "study_id", "donor_id"]
ZS_L0_PATH  = os.path.join(GF_DIR, f"glia_geneformer_zeroshot_L0{SUFFIX}.h5ad")
CPT_L0_PATH = os.path.join(GF_DIR, f"glia_geneformer_cpt_aggregated_v2_seed0_L0{SUFFIX}.h5ad")

def _extract_L0(model_dir, tag, out_h5ad):
    """Run EmbExtractor at emb_layer=0 on a checkpoint dir, save an aligned embedding h5ad."""
    ee = EmbExtractor(model_type="Pretrained", num_classes=0, emb_mode="cell",
                      max_ncells=None, emb_layer=0, emb_label=EMB_LABELS,
                      forward_batch_size=64, nproc=4)
    work = f"/content/gf_emb_{tag}{SUFFIX}"; os.makedirs(work, exist_ok=True)
    df = ee.extract_embs(model_dir, TOKENIZED_DATASET, work, f"glia_{tag}")
    emb_cols = [c for c in df.columns if c not in EMB_LABELS]
    df = df.set_index("cell_index").reindex(glia.obs["cell_index"].values)
    assert df[emb_cols].notna().all().all(), f"{tag}: rows missing after cell_index realignment"
    a = ad.AnnData(X=df[emb_cols].to_numpy(dtype=np.float32),
                   obs=glia.obs[EMB_LABELS].reset_index(drop=True))
    a.write_h5ad(out_h5ad)
    print(f"{tag} L0 -> {out_h5ad}  {a.shape}")

# zero-shot L0 (frozen base)
if os.path.exists(ZS_L0_PATH):
    print("zero-shot L0 exists, skipping:", ZS_L0_PATH)
else:
    _extract_L0(MODEL_DIR, "zeroshot_L0", ZS_L0_PATH)

# CPT L0 (base + v2 adapter, merged)
if os.path.exists(CPT_L0_PATH):
    print("CPT L0 exists, skipping:", CPT_L0_PATH)
else:
    base = BertForMaskedLM.from_pretrained(MODEL_DIR)
    merged = PeftModel.from_pretrained(base, ADAPTER_DIR).merge_and_unload()
    MERGED_DIR = f"/content/gf_v2_merged{SUFFIX}"; os.makedirs(MERGED_DIR, exist_ok=True)
    merged.save_pretrained(MERGED_DIR)
    base.config.to_json_file(os.path.join(MERGED_DIR, "config.json"))
    _extract_L0(MERGED_DIR, "cpt_L0", CPT_L0_PATH)

## 5 — Assemble the four embedding matrices

### 5a — Load L−1 (existing) and L0 (new), align all four to `cell_index`

Four matrices over the same cells: zero-shot and CPT, each at `emb_layer=-1` (loaded from the existing colab_09 / colab_11 outputs) and at `emb_layer=0` (produced above). All are realigned to the substrate's `cell_index` so a row is the same cell everywhere. Under `SMOKE`, the L−1 files are subset to the smoke cells; the L0 files already are.

In [ ]:
def _load_aligned(path):
    a = sc.read_h5ad(path)
    df = pd.DataFrame(a.X, index=a.obs["cell_index"].values)
    df = df.reindex(glia.obs["cell_index"].values)
    assert df.notna().all().all(), f"{path}: rows missing after cell_index alignment"
    return df.to_numpy(dtype=np.float32)

ZS_L1_PATH  = os.path.join(GF_DIR, "glia_geneformer_zeroshot.h5ad")                       # colab_09
CPT_L1_PATH = os.path.join(GF_DIR, "glia_geneformer_cpt_aggregated_v2_seed0.h5ad")         # colab_11 v2
for p in (ZS_L1_PATH, CPT_L1_PATH):
    assert os.path.exists(p), f"missing existing L-1 embedding {p}"

EMB = {
    ("zeroshot", "L-1"): _load_aligned(ZS_L1_PATH),
    ("cpt",      "L-1"): _load_aligned(CPT_L1_PATH),
    ("zeroshot", "L0"):  _load_aligned(ZS_L0_PATH),
    ("cpt",      "L0"):  _load_aligned(CPT_L0_PATH),
}
for k, v in EMB.items():
    print(k, v.shape)
assert len({v.shape for v in EMB.values()}) == 1, "embedding matrices differ in shape"

# shared masks / labels for the evals
obs        = glia.obs.reset_index(drop=True)
is_train   = (obs["split"] == "train").to_numpy()
is_test    = (obs["split"] == "test").to_numpy()
lineage    = obs["lineage"].to_numpy()
substate   = obs["substate"].to_numpy()
apoe       = obs["apoe_carrier"].to_numpy()
LAYERS     = ["L-1", "L0"]

def band_probe(delta_pp):    # eval #1 and eval #2 k-NN share the accuracy-gain shape
    if delta_pp < -2: return "regression"
    if delta_pp >= 10: return "decisive"
    if delta_pp >= 5:  return "meaningful"
    return "noise"

def band_knn(delta_pp):
    if delta_pp < -3: return "regression"
    if delta_pp >= 10: return "decisive"
    if delta_pp >= 5:  return "meaningful"
    return "noise"

def band_sil(delta):
    if delta < 0: return "regression"
    if delta >= 0.10: return "decisive"
    if delta >= 0.05: return "meaningful"
    return "noise"

## 6 — Eval #1: substate linear probe

### 6a — Held-out substate composition audit (a single-donor substate is a null, not a win)

The probe can only be trusted where the held-out set actually *has* the substate. The contract requires reporting, per lineage and per substate, how many held-out donors and which studies carry it — a substate that is near-absent or single-donor in the test set is unevaluable and is reported as underpowered, never as a win. The `intermediate` bucket is excluded from the binary probe (microglia: activated vs homeostatic; astrocyte: reactive vs resting) and shown here for completeness.

In [ ]:
BINARY = {"microglia": ("homeostatic", "activated"), "astrocyte": ("resting", "reactive")}

print("held-out substate composition (donors / studies driving eval #1):")
for lin, (neg, pos) in BINARY.items():
    m = is_test & (lineage == lin)
    sub = obs.loc[m, ["substate", "donor_id", "study_id"]]
    print(f"\n[{lin}] binary classes = {neg} vs {pos}  (intermediate excluded)")
    for s in (neg, pos, "intermediate"):
        ss = sub[sub["substate"] == s]
        nd = ss["donor_id"].nunique()
        studies = ss["study_id"].value_counts().to_dict()
        flag = "" if s == "intermediate" else ("  [UNDERPOWERED]" if nd < 3 else "")
        print(f"  {s:12s}: {len(ss):5d} cells | {nd} donors | {studies}{flag}")

### 6b — Probe at both extraction points, Δ vs zero-shot with contract bands

A logistic-regression probe is trained on the train-split cells' embedding and scored (balanced accuracy) on the disjoint held-out donors — donor-held-out by construction, so a prediction cannot be a memorised donor identity. The same probe is fit separately on the zero-shot and CPT embeddings at each layer; the eval verdict is the **CPT − zero-shot** gain against the fixed bands (noise ca. 2% / meaningful ≥5% / decisive ≥10% / regression >2% drop). The L−1-vs-L0 comparison shows whether reading one layer deeper — where layer 11's absorbed gain lives — surfaces structure the standard readout misses.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import balanced_accuracy_score

def probe_bacc(X, y_train_mask, y_test_mask, y):
    sc_ = StandardScaler().fit(X[y_train_mask])
    clf = LogisticRegression(max_iter=2000, class_weight="balanced")
    clf.fit(sc_.transform(X[y_train_mask]), y[y_train_mask])
    pred = clf.predict(sc_.transform(X[y_test_mask]))
    return balanced_accuracy_score(y[y_test_mask], pred)

EVAL1 = {}
for lin, (neg, pos) in BINARY.items():
    in_lin  = lineage == lin
    is_bin  = np.isin(substate, [neg, pos])
    tr_mask = is_train & in_lin & is_bin
    te_mask = is_test  & in_lin & is_bin
    y = (substate == pos).astype(int)   # 1 = activated / reactive
    print(f"\n[{lin}] train {int(tr_mask.sum())} / test {int(te_mask.sum())} cells")
    for layer in LAYERS:
        b_zs  = probe_bacc(EMB[("zeroshot", layer)], tr_mask, te_mask, y)
        b_cpt = probe_bacc(EMB[("cpt", layer)],      tr_mask, te_mask, y)
        delta = (b_cpt - b_zs) * 100
        EVAL1[(lin, layer)] = {"bacc_zeroshot": round(float(b_zs), 4),
                               "bacc_cpt": round(float(b_cpt), 4),
                               "delta_pp": round(float(delta), 2),
                               "verdict": band_probe(delta)}
        print(f"  {layer}: zero-shot {b_zs:.3f} -> CPT {b_cpt:.3f} | Δ {delta:+.2f} pp | {band_probe(delta)}")

## 7 — Eval #2: APOE-carrier recovery (Stanton core)

### 7a — APOE composition and confound audit (study × region of the held-out carriers)

The load-bearing biological axis: does CPT make E4-carrier status more recoverable *within* each lineage? First the mandatory confound audit — a "recovery" that is really study or region leakage is a null. Carrier vs non-carrier is binary; `e2` cells are excluded per the locked label definition. This cell reports, per lineage, the held-out carrier / non-carrier counts broken down by study and region, and flags any lineage where a class is too thin to evaluate.

In [ ]:
APOE_KEEP = {"carrier", "noncarrier"}   # e2 excluded per locked E4 definition

print("held-out APOE composition (study x region driving eval #2):")
for lin in ("microglia", "astrocyte"):
    m = is_test & (lineage == lin) & np.isin(apoe, list(APOE_KEEP))
    sub = obs.loc[m, ["apoe_carrier", "study_id", "region", "donor_id"]]
    print(f"\n[{lin}] held-out carrier/noncarrier cells: {len(sub)}")
    for cls in ("carrier", "noncarrier"):
        cc = sub[sub["apoe_carrier"] == cls]
        nd = cc["donor_id"].nunique()
        flag = "  [UNDERPOWERED]" if nd < 3 else ""
        print(f"  {cls:11s}: {len(cc):5d} cells | {nd} donors{flag}")
    print("  study x region:")
    print(sub.groupby(["study_id", "region"], observed=True).size().to_string())

### 7b — Silhouette + k-NN within astro / within micro, at both extraction points

Two metrics, reported together per the contract. **k-NN** fits on the train-split cells and predicts held-out carrier status (balanced accuracy, donor-disjoint) — this is the **load-bearing** metric here. **Silhouette** of the binary E4 label measures how separated carriers and non-carriers are in the held-out embedding, but at these donor counts it is computed on few cells and is scale- and density-sensitive, so it is treated as **corroborating-only**: it is still scored against its band, but a silhouette move is not read as a verdict on its own — it either agrees with the k-NN read or it is noise. Both are scored as CPT − zero-shot against their own bands — silhouette (noise 0.02 / meaningful ≥0.05 / decisive ≥0.10 / regression Δ<0) and k-NN (noise 3–5% / meaningful ≥5% / decisive ≥10% / regression >3% drop) — at L−1 and L0. Given the thin held-out donor counts (carrier 7 / non-carrier 10 across the whole test set), wide swings are expected and N=3 seeds will not narrow them; the composition audit above governs what is even evaluable.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import silhouette_score

def apoe_metrics(X, tr_mask, te_mask, y):
    sc_ = StandardScaler().fit(X[tr_mask])
    Xtr, Xte = sc_.transform(X[tr_mask]), sc_.transform(X[te_mask])
    knn = KNeighborsClassifier(n_neighbors=15).fit(Xtr, y[tr_mask])
    bacc = balanced_accuracy_score(y[te_mask], knn.predict(Xte))
    sil = silhouette_score(Xte, y[te_mask]) if len(np.unique(y[te_mask])) == 2 else float("nan")
    return bacc, sil

EVAL2 = {}
for lin in ("microglia", "astrocyte"):
    in_lin = (lineage == lin) & np.isin(apoe, list(APOE_KEEP))
    tr_mask = is_train & in_lin
    te_mask = is_test  & in_lin
    y = (apoe == "carrier").astype(int)
    print(f"\n[{lin}] train {int(tr_mask.sum())} / test {int(te_mask.sum())} cells")
    for layer in LAYERS:
        k_zs, s_zs   = apoe_metrics(EMB[("zeroshot", layer)], tr_mask, te_mask, y)
        k_cpt, s_cpt = apoe_metrics(EMB[("cpt", layer)],      tr_mask, te_mask, y)
        dk = (k_cpt - k_zs) * 100
        ds = s_cpt - s_zs
        EVAL2[(lin, layer)] = {
            "knn_bacc_zeroshot": round(float(k_zs), 4), "knn_bacc_cpt": round(float(k_cpt), 4),
            "knn_delta_pp": round(float(dk), 2), "knn_verdict": band_knn(dk),
            "sil_zeroshot": round(float(s_zs), 4), "sil_cpt": round(float(s_cpt), 4),
            "sil_delta": round(float(ds), 4), "sil_verdict": band_sil(ds)}
        print(f"  {layer}: k-NN {k_zs:.3f}->{k_cpt:.3f} (Δ{dk:+.2f}pp {band_knn(dk)}) | "
              f"sil {s_zs:+.3f}->{s_cpt:+.3f} (Δ{ds:+.3f} {band_sil(ds)})")

## 8 — Summary and handoff

### 8a — Verdict table, append the audit trace, print the commit commands

The per-eval, per-lineage, per-layer verdicts assembled into one table, then written to `outputs/audit_report.json` under `geneformer_cpt_evals`. The headline read is the **L−1 vs L0** contrast: L−1 is the verdict the pipeline's actual readout supports; the L−1→L0 change is the partial (layer-11-only, head-blind) eval-space view of the absorbed gain. A `SMOKE` run does **not** write the audit trace (its numbers are plumbing, not results). Commit commands are printed for the WSL side, since Colab has no git credentials.

In [ ]:
import shlex

print("=== EVAL #1 (substate probe) — balanced-acc gain, CPT vs zero-shot ===")
for (lin, layer), r in EVAL1.items():
    print(f"  {lin:10s} {layer:4s}: {r['bacc_zeroshot']:.3f} -> {r['bacc_cpt']:.3f}  Δ{r['delta_pp']:+.2f}pp  [{r['verdict']}]")
print("\n=== EVAL #2 (APOE recovery) — k-NN gain / silhouette gain ===")
for (lin, layer), r in EVAL2.items():
    print(f"  {lin:10s} {layer:4s}: kNN Δ{r['knn_delta_pp']:+.2f}pp [{r['knn_verdict']}] | "
          f"sil Δ{r['sil_delta']:+.3f} [{r['sil_verdict']}]")

if SMOKE:
    print("\n[SMOKE] audit trace NOT written (plumbing run).")
else:
    AUDIT_REPORT_PATH = os.path.join(REPO_PATH, "outputs", "audit_report.json")
    with open(AUDIT_REPORT_PATH) as f:
        report = json.load(f)
    report["geneformer_cpt_evals"] = {
        "status": "computed", "date": TODAY, "regime": "aggregated", "seed": SEED,
        "reads_run": "geneformer_cpt_aggregated_v2", "extraction_points": {"L-1": "second-to-last", "L0": "last layer"},
        "geneformer_commit": GENEFORMER_COMMIT,
        "eval1_substate_probe": {f"{lin}|{layer}": r for (lin, layer), r in EVAL1.items()},
        "eval2_apoe_recovery":  {f"{lin}|{layer}": r for (lin, layer), r in EVAL2.items()},
        "embedding_files": {
            "zeroshot_L-1": os.path.relpath(ZS_L1_PATH, DRIVE_ROOT),
            "cpt_L-1":      os.path.relpath(CPT_L1_PATH, DRIVE_ROOT),
            "zeroshot_L0":  os.path.relpath(ZS_L0_PATH, DRIVE_ROOT),
            "cpt_L0":       os.path.relpath(CPT_L0_PATH, DRIVE_ROOT)},
        "note": ("L-1 = pipeline readout; L0 captures layer-11's absorbed share only (head unreachable as "
                 "embedding). Eval #2: k-NN is load-bearing; silhouette corroborating-only at these donor counts."),
    }
    with open(AUDIT_REPORT_PATH, "w") as f:
        json.dump(report, f, indent=2)
    print("\naudit trace appended ->", AUDIT_REPORT_PATH)
    rel = os.path.relpath(AUDIT_REPORT_PATH, REPO_PATH)
    print("\n=== Commit + push (from WSL) ===")
    print(f"  git add {shlex.quote(rel)}")
    print("  git commit -m 'colab_12: Geneformer CPT evals #1 (substate probe) + #2 (APOE) at L-1 and L0'")
    print("  git push")